<!-- SPDX-License-Identifier: CC-BY-4.0 -->
<!-- Copyright Contributors to the Egeria project. -->

# 04a Simulation-case layout: a parametric FEA sweep

Coco Pharmaceuticals' device-engineering team runs finite-element (FEA) parameter sweeps. A **sweep** is the set of cases made from one template
over one declared grid; here it is one folder, `study-<name>/`. A **case** is ONE grid point (a modulus E, a thickness t and a mesh level), with
its own folder; it is a part of the sweep, never the sweep itself. A **run** is one attempt to execute a case; its outputs are a **result
set**. The **study**, the investigation itself (goal, owner, decisions), is not visible from file names.

A sweep produces many files a reader does not need to see: solver results, status files, logs, restart files, scratch and backups. This notebook
generates a synthetic CalculiX-shaped sweep. Only the file-name conventions are modelled: every file is a one-line placeholder, and no solver is
run.

The notebook recognizes the tree first with the `generic-relevance` profile alone ("N assets"), then with the `fea-sweep` profile ("1 sweep, 60
cases, X% hidden output"). It builds a parameter table from the case names, checks conformance against a declared grid, and reports drift after
cases are added. It answers four questions: which cases exist, which ran, which are off-grid, and where the results are.

Only names are used, so a case "ran" when it holds the results file `job.frd`: a failed attempt that wrote results looks the same as a good one,
and because the solver writes `job.*` in place, a re-run overwrites the previous attempt and is invisible. Run attempts are not modelled in
`fea-sweep` 0.1.0.

It works locally with Python and PyYAML only. Nothing is fetched and nothing is written to Egeria. The Resource Explorer, Portal, Egeria and
lineage cells are marked "Phase 3/4 — not run". The run, result and validation lineage (UC7) belongs to notebook 04.

## Set up

The recognizer lives in `layout-profiles/` of this repository, two folders up from this notebook. The work folder is a fresh temporary folder.

In [ ]:
import json
import sys
import tempfile
from collections import Counter
from pathlib import Path

LAYOUT = (Path.cwd() / ".." / ".." / "layout-profiles").resolve()
assert (LAYOUT / "recognizer" / "recognize.py").is_file(), "run this notebook from its own folder in a checkout of egeria-workspaces"
sys.path.insert(0, str(LAYOUT))
sys.path.insert(0, str(LAYOUT / "samples"))

import fea_sweep_gen
from recognizer import conformance, drift
from recognizer.layout_profile import load_profile
from recognizer.recognize import Recognizer, load_instance

GENERIC = load_profile("generic-relevance", LAYOUT / "profiles")
FEA = load_profile("fea-sweep", LAYOUT / "profiles")
WORK = Path(tempfile.mkdtemp(prefix="nb04a-"))
SEED = 1
print(f"profiles {GENERIC.profile_id} {GENERIC.version} and {FEA.profile_id} {FEA.version}; work folder made")


def write_tree(name, files, truth=()):
    """Write a sweep to WORK/name: study-<name>/, paths.tsv and truth.jsonl. The scanned root is WORK/name, the parent of the study folder."""
    fea_sweep_gen.write_outputs(files, list(truth), WORK / name)
    return WORK / name / "paths.tsv"


def survey(name, profile, instance=None):
    """Recognize one tree by names and sizes. With an instance file, its declared grid is checked too (the only content read)."""
    out = WORK / f"{name}-{profile.profile_id}"
    rec = Recognizer(profile, instance=load_instance(instance, profile)[0] if instance else None)
    rec.instance_file = load_instance(instance, profile)[1] if instance else None
    classes = rec.run(WORK / name / "paths.tsv", out)
    return classes, out


def findings(out):
    return [json.loads(line) for line in (out / "findings.jsonl").read_text().splitlines()]

## 1. Generate a clean sweep

The default grid: 4 moduli x 5 thicknesses x 3 mesh levels = 60 cases, of which 80% ran. The seed picks the values, the study name and the cases
that ran.

In [ ]:
layout = fea_sweep_gen.layout(SEED)
files, _ = fea_sweep_gen.generate(SEED)
write_tree("clean", files)
STUDY = layout["study"]
print(f"{STUDY}: {len(files)} files")
print("E:", layout["E"], "| t:", [fea_sweep_gen.fmt_t(t) for t in layout["t"]], "| mesh:", layout["mesh"])
top = Counter(p.split("/")[1] for p in files)
for name, n in sorted(top.items()):
    print(f"  {name:<12} {n:>4} file(s)")
one = layout["ran"][0]
print(f"one case that ran, {one}:", sorted(p.rsplit("/", 1)[1] for p in files if f"/cases/{one}/" in p))

## 2. Generic relevance alone: "N assets"

`generic-relevance` knows nothing about simulations. It hides `.log` and `.bak` files and names no component, so a reader sees every other file.

In [ ]:
g, g_out = survey("clean", GENERIC)
hidden = g["by_relevance"].get("hidden", {"files": 0})["files"]
folders = len({p.rsplit("/", 1)[0] for p in files})
print(f"generic-relevance: {g['total_files'] - hidden} assets shown in {folders} folders; "
      f"{g['components']['top_level']} components; {g['shares']['hidden_files']:.1%} hidden")

## 3. The `fea-sweep` profile: "1 sweep, 60 cases, X% hidden output"

The folder that holds `sweep.yaml` is ONE component, the sweep; every case folder under its `cases/` is a part of it. The result sets of the
runs (`.frd`, `.dat`, `.12d`), status files (`.sta`, `.cvg`, `.msg`), run logs, restart files (`.rout`, for a restart: a run that continues from
another), scratch and backups are hidden: counted, not shown.

In [ ]:
c, c_out = survey("clean", FEA)
print(f"{c['components']['top_level']} sweep, {c['components']['parts']} cases, {c['shares']['hidden_files']:.0%} hidden output "
      f"({c['shares']['hidden_bytes']:.0%} of the bytes)")
print("hidden, per rule (files):")
for rule, v in sorted(c["by_rule"].items()):
    if rule in ("R-fea-results", "R-fea-status", "R-fea-run-log", "R-fea-restart", "R-fea-scratch", "R-fea-backup"):
        print(f"  {rule:<16} {v['files']:>4}")
print("shown:", c["total_files"] - c["by_relevance"]["hidden"]["files"], "files")

## 4. A parameter table from the names

Each case's part finding carries the tokens of its folder name (`case_E<E>_t<t>_mesh-<level>`, with `p` for the decimal point). No file is opened.

In [ ]:
parts = [f for f in findings(c_out) if f.get("grain") == "component" and f.get("part_of")]
tokens = [f["tokens"] for f in parts if "tokens" in f]
print(f"{len(tokens)} of {len(parts)} cases have parameter tokens")
ts = sorted({t["t"] for t in tokens}, key=lambda v: float(v.replace("p", ".")))
print("E \\ t".ljust(8) + "".join(t.rjust(7) for t in ts))
for e in sorted({t["E"] for t in tokens}, key=int):
    row = [sum(1 for x in tokens if x["E"] == e and x["t"] == t) for t in ts]
    print(e.ljust(8) + "".join(str(n).rjust(7) for n in row))
print("(each cell: the number of mesh levels)")

## 5. The declared grid and conformance

The grid comes from the PROFILE INSTANCE, a small YAML file the notebook writes in the work folder. It is never read from `sweep.yaml`, which is
never opened. Each case needs its input deck `job.inp`, and a case that ran (it holds `job.frd`) needs `job.sta`. A case without `job.frd` is
counted as `not-run`: a state, not a finding.

In [ ]:
INSTANCE = WORK / "fea-sweep.instance.yaml"
fea_sweep_gen.write_instance(fea_sweep_gen.instance(layout), INSTANCE)
print(INSTANCE.read_text())
clean, clean_out = survey("clean", FEA, INSTANCE)
cj = conformance.write(clean_out)
print(conformance.headline(cj))
print("not-run:", clean["conformance"]["states"]["not-run"]["count"], "of", clean["conformance"]["states"]["not-run"]["checked"], "cases")
print("content read:", (clean_out / "content-reads.txt").read_text().strip())

## 6. Inject deviations and survey again

`samples/specs/fea-a.yaml` asks the generator for one deviation of each class: a missing deck, results without a deck, two misnamed case folders,
an off-grid thickness, a duplicate parameter combination, a stray deck, a mesh level outside the grid, an unexpected top-level folder and a post
folder with no case. The generator writes what it injected to `truth.jsonl`; the survey does not see that file.

In [ ]:
files_m, truth = fea_sweep_gen.generate(SEED, spec=fea_sweep_gen.load_spec(LAYOUT / "samples" / "specs" / "fea-a.yaml"))
write_tree("changed", files_m, truth)
changed, changed_out = survey("changed", FEA, INSTANCE)
mj = conformance.write(changed_out)
print(conformance.headline(mj))
d = drift.write(clean_out, changed_out)
print(drift.headline(d))
for f in d["new"]:
    print(f"  {f['kind']:<10} {f['status']:<10} {f['path'][len(STUDY) + 1:]}  ({f['rule_id']}{', ' + f['reason'] if f.get('reason') else ''})")
injected = sorted((t["path"], t["kind"], t["status"]) for t in truth)
found = sorted((f["path"], f["kind"], f["status"]) for f in d["new"])
print(f"{len(injected)} injected, {len(found)} new findings, equal: {injected == found}")
assert injected == found

## 7. The E1 layout questions, answered from the stored outputs

Only `findings.jsonl`, `classes.json` and `conformance.json` of the changed survey are read here. "Ran" means the case folder holds `job.frd`
(names only; the outcome of the run is not known).

In [ ]:
fs = findings(changed_out)
cls = json.loads((changed_out / "classes.json").read_text())
cj = json.loads((changed_out / "conformance.json").read_text())
cases = sorted(f["path"] for f in fs if f.get("kind") == "simulation-case")
not_run = set(cls["conformance"]["states"]["not-run"]["paths"])
typed = {f["path"] for f in fs if f.get("kind") == "simulation-case" and "tokens" in f}
ran = sorted(p for p in typed if p not in not_run)
off_grid = sorted(f["path"] for f in cj["findings"] if f["status"] == "off-grid")
results = Counter(f["path"] for f in fs if f.get("rule_id") == "R-fea-results")

print(f"Which cases exist?      {len(cases)} case folders ({len(typed)} named by the grammar, {len(cases) - len(typed)} misnamed)")
print(f"Which ran?              {len(ran)} of the {len(typed)} named cases hold job.frd (outcome unknown); not-run: {len(not_run)} (misnamed folders are not judged)")
print(f"Which are off-grid?     {[p.rsplit('/', 1)[1] for p in off_grid]}")
print(f"Where are the results?  in {len(results)} case folders (misnamed ones included): job.frd, job.dat (and job.12d), "
      f"{sum(f['files'] for f in fs if f.get('rule_id') == 'R-fea-results')} files, hidden by default; for example {sorted(results)[0][len(STUDY) + 1:]}")
print(f"Which are duplicates?   {[f['path'].rsplit('/', 1)[1] for f in cj['findings'] if f['status'] == 'duplicate']}")

## 8. Drift after adding cases

The sweep grows: 6 cases at a new modulus (declared in the instance), 2 cases removed and 1 case moved to another grid point. Each survey is a
snapshot, written once; drift compares two snapshots. Parts are compared by finding id, which is made from the path: a renamed case is one
resolved part and one new part, not a move.

In [ ]:
later, e_new, renames = fea_sweep_gen.evolve(files, layout, SEED)
fea_sweep_gen.write_instance(fea_sweep_gen.instance(layout, (e_new,)), INSTANCE)
write_tree("later", later)
before, before_out = survey("clean", FEA, INSTANCE)
after, after_out = survey("later", FEA, INSTANCE)
dd = drift.write(before_out, after_out)
print(f"{before['components']['parts']} -> {after['components']['parts']} cases; conformance drift: {drift.headline(dd)}")
p = dd["parts"]
print(f"parts: {len(p['new'])} new, {len(p['resolved'])} resolved, {p['unchanged']} unchanged")
print("renamed:", renames)
print("new:     ", [x["path"].rsplit("/", 1)[1] for x in p["new"]])
print("resolved:", [x["path"].rsplit("/", 1)[1] for x in p["resolved"]])

## Phase 3/4 — not run

In Phase 3 the Resource Explorer will make the same survey of a governed tree, and the Portal will show the sweep, its cases and the findings for
accept-or-fix decisions. The findings could be recorded in Egeria. In Phase 4, notebook 04 will link each run attempt to its result set and its
validation (UC7, lineage); choosing the current (approved) attempt and sealing finished attempts are not built.

None of this is done here. The cells below only show the steps; they do nothing.

In [ ]:
# Phase 3/4 — not run: the Resource Explorer survey of a governed sweep with the fea-sweep profile and its instance attached.
PHASE34 = False
if PHASE34:
    raise RuntimeError("Phase 3/4 — not run")
print("Phase 3/4 — not run: the Resource Explorer survey of a governed sweep with the fea-sweep profile and its instance attached")

In [ ]:
# Phase 3/4 — not run: Portal curation (the sweep, its cases, the conformance score and the findings list).
if PHASE34:
    raise RuntimeError("Phase 3/4 — not run")
print("Phase 3/4 — not run: Portal curation")

In [ ]:
# Phase 3/4 — not run: record the sweep, its cases and the findings in Egeria (the layout profile types wait for Gate 2).
if PHASE34:
    raise RuntimeError("Phase 3/4 — not run")
print("Phase 3/4 — not run: record the sweep, its cases and the findings in Egeria")

In [ ]:
# Phase 3/4 — not run: run -> result -> validation lineage (UC7, notebook 04, Phase 4).
if PHASE34:
    raise RuntimeError("Phase 3/4 — not run")
print("Phase 3/4 — not run: run -> result -> validation lineage")

## Clean up

In [ ]:
import shutil

shutil.rmtree(WORK)
print("work folder removed")